In [1]:
!pip install -q Flask google-genai

In [2]:
import os
from flask import Flask

In [3]:
import time

def generate_content(prompt):

    model = genai.GenerativeModel(
        "gemini-3.5-flash"
    )

    for attempt in range(3):

        try:
            response = model.generate_content(prompt)

            if response.text:
                return response.text

            raise Exception("Empty response from Gemini.")

        except Exception as e:

            error_message = str(e)

            if "503" in error_message or "UNAVAILABLE" in error_message:

                if attempt < 2:
                    time.sleep(4)
                    continue

            raise e

In [4]:
!pkill -f "python app_flask.py"

In [5]:
import subprocess

subprocess.Popen(
    ["python", "app_flask.py"]
)

<Popen: returncode: None args: ['python', 'app_flask.py']>

In [6]:
%%writefile app_flask.py
from flask import Flask, render_template_string, request, jsonify
import os
from google import genai

app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>AI Content Generator (Flask)</title>
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/bootstrap/5.3.0/css/bootstrap.min.css">
    <style>
        body { background-color: #f8f9fa; font-family: sans-serif; }
        .container { max-width: 800px; margin-top: 50px; }
        .card { padding: 30px; border-radius: 15px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        .output-box { background: #ffffff; border: 1px solid #ced4da; padding: 20px; border-radius: 10px; margin-top: 20px; white-space: pre-wrap; }
    </style>
</head>
<body>
    <div class="container">
        <div class="card">
            <h2 class="text-center mb-4">✨ AI Content Generator (Flask)</h2>
            <form id="gen-form">
                <div class="mb-3">
                    <label class="form-label">Gemini API Key</label>
                    <input type="password" id="api_key" class="form-control" required placeholder="Enter your API key">
                </div>
                <div class="mb-3">
                    <label class="form-label">Topic</label>
                    <textarea id="topic" class="form-control" rows="3" required placeholder="Example: Artificial Intelligence in Education"></textarea>
                </div>
                <div class="row">
                    <div class="col-md-6 mb-3">
                        <label class="form-label">Tone</label>
                        <select id="tone" class="form-select">
                            <option>Professional</option>
                            <option>Casual</option>
                            <option>Friendly</option>
                            <option>Creative</option>
                        </select>
                    </div>
                    <div class="col-md-6 mb-3">
                        <label class="form-label">Length</label>
                        <select id="length" class="form-select">
                            <option value="Short">Short (~150 words)</option>
                            <option value="Medium">Medium (~350 words)</option>
                            <option value="Long">Long (~700 words)</option>
                        </select>
                    </div>
                </div>
                <button type="submit" class="btn btn-primary w-100">Generate Content</button>
            </form>

            <div id="result-section" style="display:none;">
                <h4 class="mt-4">📄 Generated Content</h4>
                <div class="output-box" id="output-content"></div>
            </div>
        </div>
    </div>

    <script>
        document.getElementById('gen-form').addEventListener('submit', async function(e) {
            e.preventDefault();
            const btn = e.target.querySelector('button');
            btn.disabled = true;
            btn.innerText = 'Generating...';

            const response = await fetch('/generate', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({
                    api_key: document.getElementById('api_key').value,
                    topic: document.getElementById('topic').value,
                    tone: document.getElementById('tone').value,
                    length: document.getElementById('length').value
                })
            });

            const data = await response.json();
            btn.disabled = false;
            btn.innerText = 'Generate Content';

            if(data.success) {
                document.getElementById('output-content').innerText = data.content;
                document.getElementById('result-section').style.display = 'block';
            } else {
                alert('Error: ' + data.error);
            }
        });
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/generate', methods=['POST'])
def generate():
    data = request.json
    api_key = data.get('api_key')
    topic = data.get('topic')
    tone = data.get('tone')
    length_val = data.get('length')

    try:
        client = genai.Client(api_key=api_key)
        prompt = f"Write a {tone.lower()} content piece about '{topic}'. Target length is {length_val.lower()}. Keep it engaging."
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=prompt
        )
        return jsonify(success=True, content=response.text)
    except Exception as e:
        return jsonify(success=False, error=str(e))

if __name__ == '__main__':
    app.run(port=5000, host='0.0.0.0')


Writing app_flask.py


In [7]:
from google.colab import output

flask_url = output.eval_js(
    "google.colab.kernel.proxyPort(5000)"
)

print("Flask App is running!")
print(flask_url)

Flask App is running!
https://5000-m-s-kkb-usw3b0-25kq2obd9tj1e-b.us-west3-0.prod.colab.dev
